### [1단계] 원본 데이터 로드 및 구조 확인 코드

In [1]:
import pandas as pd

# 파일 경로 및 Excel 파일 객체 생성
file_path = '../../data/raw/books_raw.xlsx'
xls = pd.ExcelFile(file_path)

# 시트 이름 확인
print("1. 시트 목록:", xls.sheet_names)

# 첫 번째 시트 데이터 불러오기
df_raw = pd.read_excel(file_path, sheet_name=xls.sheet_names[0])

# 데이터 크기 및 상위 5개 행 확인
print(f"2. 전체 데이터 크기: {df_raw.shape}")
print("\n3. 컬럼 목록:", df_raw.columns.tolist())

# 상위 데이터 미리보기 (순위, 상품번호, 상품명)
display(df_raw[['순위', '상품번호', '상품명', '저자']].head())

1. 시트 목록: ['Sheet_국내도서']
2. 전체 데이터 크기: (200, 11)

3. 컬럼 목록: ['순위', 'ISBN', '상품번호', '상품명', '판매가', 'YES포인트', '저자', '출판사', '설명', '출고예상일', '관리분류']


,순위,상품번호,상품명,저자
0,1,139807878,빨강의 자서전,앤 카슨 저/민승남 역
1,2,195842609,트렌드 코리아 2027,"김난도,전미영,최지혜,권정윤,한다혜,이혜원,전다현,이수진,서유현,이준영,이향은,김나..."
2,3,197068949,가호,무라카미 하루키 저/홍은주 역
3,4,197772287,가짜 돈 진짜 부,박종훈 저
4,5,196557643,2027 공단기 소방단기 써니 행정법총론 기출문제집,박준철 편저


### [2단계] 상품번호 기반 상세 페이지 URL 생성 및 단건 테스트 코드

In [20]:
import time
import json
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os

# 1. 원본 전체 데이터 로드
file_path = '../../data/raw/books_raw.xlsx'
df_raw = pd.read_excel(file_path)
print(f"전체 수집 대상 도서 수: {len(df_raw)}권\n")

ext_data = []
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 2. 전체 도서 순회하며 상세 페이지 크롤링
for idx, row in df_raw.iterrows():
    goods_no = row['상품번호']
    title = row['상품명']
    isbn = row['ISBN']
    
    url = f"https://www.yes24.com/Product/Goods/{goods_no}"
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            rating = "정보 없음"
            review_count = "정보 없음"
            sales_index = "정보 없음"
            
            # A. JSON-LD에서 평점 및 리뷰 수 추출
            script_tag = soup.find('script', type='application/ld+json')
            if script_tag and script_tag.string:
                try:
                    ld_data = json.loads(script_tag.string.strip())
                    agg_rating = ld_data.get('aggregateRating', {})
                    if agg_rating:
                        rating = agg_rating.get('ratingValue', "정보 없음")
                        review_count = agg_rating.get('reviewCount', "정보 없음")
                except Exception:
                    pass
            
            # B. 판매지수 추출
            sale_el = soup.select_one('.saleNum')
            if sale_el:
                sale_match = re.search(r'([\d,]+)', sale_el.get_text())
                if sale_match:
                    sales_index = sale_match.group(1).replace(',', '')
            else:
                text_content = soup.get_text()
                sales_match = re.search(r'판매지수[\s\xa0]*([\d,]+)', text_content)
                if sales_match:
                    sales_index = sales_match.group(1).replace(',', '')
            
            ext_data.append({
                '상품번호': goods_no,
                'ISBN': isbn,
                '상품명': title,
                '리뷰점수': rating,
                '리뷰건수': review_count,
                '판매지수': sales_index
            })
            print(f"[{idx+1}/{len(df_raw)}] 성공: {title} (평점: {rating}, 리뷰수: {review_count}, 판매지수: {sales_index})")
        else:
            print(f"[{idx+1}/{len(df_raw)}] 실패 (HTTP {response.status_code}): {title}")
            ext_data.append({'상품번호': goods_no, 'ISBN': isbn, '상품명': title, '리뷰점수': '오류', '리뷰건수': '오류', '판매지수': '오류'})
    except Exception as e:
        print(f"[{idx+1}/{len(df_raw)}] 에러: {title} ({e})")
        ext_data.append({'상품번호': goods_no, 'ISBN': isbn, '상품명': title, '리뷰점수': '예외', '리뷰건수': '예외', '판매지수': '예외'})
    
    # 서버 부하 방지 0.3초 대기
    time.sleep(0.3)

# 3. CSV 저장
df_ext = pd.DataFrame(ext_data)
os.makedirs('../../data/raw', exist_ok=True)
ext_file_path = '../../data/raw/books_raw_ext.csv'
df_ext.to_csv(ext_file_path, index=False, encoding='utf-8-sig')
print(f"\n[전체 크롤링 완료] 저장 경로: {ext_file_path}")

전체 수집 대상 도서 수: 200권

[1/200] 성공: 빨강의 자서전 (평점: 9.6, 리뷰수: 9, 판매지수: 132366)
[2/200] 성공: 트렌드 코리아 2027 (평점: 9.4, 리뷰수: 77, 판매지수: 249162)
[3/200] 성공: 가호 (평점: 10.0, 리뷰수: 10, 판매지수: 235206)
[4/200] 성공: 가짜 돈 진짜 부 (평점: 10.0, 리뷰수: 1, 판매지수: 80070)
[5/200] 성공: 2027 공단기 소방단기 써니 행정법총론 기출문제집 (평점: 9.7, 리뷰수: 43, 판매지수: 240237)
[6/200] 성공: 쇼펜하우어 인생수업 (30만 부 기념 개정증보판) (평점: 8.9, 리뷰수: 280, 판매지수: 300552)
[7/200] 성공: 세네카, 오늘을 빼앗기고 있는 당신에게 (평점: 8.2, 리뷰수: 255, 판매지수: 490788)
[8/200] 성공: 싯다르타 (평점: 9.4, 리뷰수: 1035, 판매지수: 445071)
[9/200] 성공: 남편의 아름다움 (평점: 8.0, 리뷰수: 4, 판매지수: 56478)
[10/200] 성공: 전지적 독자 시점 문고본 세트 (평점: 10.0, 리뷰수: 5, 판매지수: 51660)
[11/200] 성공: 모순 (평점: 9.5, 리뷰수: 2852, 판매지수: 459831)
[12/200] 성공: 에로스, 달콤씁쓸한 (평점: 10.0, 리뷰수: 1, 판매지수: 41790)
[13/200] 성공: 브레인 악셀 (평점: 9.3, 리뷰수: 189, 판매지수: 91875)
[14/200] 성공: 니체의 초월자 (평점: 9.7, 리뷰수: 263, 판매지수: 442554)
[15/200] 성공: 머니 트렌드 2027 (평점: 9.7, 리뷰수: 39, 판매지수: 132570)
[16/200] 성공: 아주 작은 습관의 힘 (50만 부 기념 스페셜 에디션) (평점: 9.4, 리뷰수: 1118, 판매지수: 220119)
[17/200] 성공: 옥스브리지의 철학 수업 (평점: 9

In [21]:
# 1. 원본 및 외부 데이터 로드
df_raw = pd.read_excel('../../data/raw/books_raw.xlsx')
df_ext = pd.read_csv('../../data/raw/books_raw_ext.csv')

# 2. 상품번호 기준 Merge
df_total = pd.merge(df_raw, df_ext[['상품번호', '리뷰점수', '리뷰건수', '판매지수']], on='상품번호', how='left')

# 3. 전체 통합 파일 저장
total_file_path = '../../data/raw/books_raw_total.csv'
df_total.to_csv(total_file_path, index=False, encoding='utf-8-sig')

print(f"[병합 완료] 총 컬럼 수: {df_total.shape[1]}, 총 행 수: {df_total.shape[0]}")
display(df_total[['순위', '상품명', '리뷰점수', '리뷰건수', '판매지수']].head(5))

[병합 완료] 총 컬럼 수: 14, 총 행 수: 200


,순위,상품명,리뷰점수,리뷰건수,판매지수
0,1,빨강의 자서전,9.6,9,132366
1,2,트렌드 코리아 2027,9.4,77,249162
2,3,가호,10.0,10,235206
3,4,가짜 돈 진짜 부,10.0,1,80070
4,5,2027 공단기 소방단기 써니 행정법총론 기출문제집,9.7,43,240237
